# Project 1, CPSC 585
## Authors: Brian Chu (brianmchu42@csu.fullerton.edu), Jannik Hofmann (jhof@csu.fullerton.edu), Gabriel Villaruel ()

### 1. Clustering (70 points)

#### (a) Data preparation and preprocessing (5 points)
Standardize (z-score scaling) the four chemical properties and save them in a file named
“StdGasProperties.csv”, which will be used for the remaining tasks. Briefly explain:
* The importance of data standardization
* Feature mean and standard deviation before and after standardization

Clustering must be performed using only the four chemical features (excluding idx).

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

df = pd.read_csv("GasProperties.csv")

# skip Idx and only scale the four chemical properties
props = ['T', 'P', 'TC', 'SV']
idx = ['Idx']

preprocessor = ColumnTransformer(
    transformers = [
        ('scaler', StandardScaler(), props),
    ],
    remainder = 'passthrough'
)

transformed = preprocessor.fit_transform(df)

transformed_df = pd.DataFrame(transformed, columns = props + idx)

print('### ORIGINAL ###')
print(df)

print('### MEAN ###')
print(df.mean())
print('### STD ###')
print(df.std())

print('### TRANSFORMED ###')
print(transformed_df)

print('### MEAN ### ')
print(transformed_df.mean())
print('### STD ###')
print(transformed_df.std())

transformed_df.to_csv('StdGasProperties.csv', index=False)

Data standardization is important to regularize the data - numbers that are large in magnitude can be harder to process with some models. For example, since the magnitude of the `T` and `SV` columns in the data are more than two orders of magnitude larger than the `P` and `TC` columns, the `T` and `SV` columns can have an outsize effect on model predictions. Regularizing ensures that all relevant features are considered equally by the model.

As seen in the printout of the data, the original data has some particular mean and standard deviation, but when regularized, the mean becomes 0 and the standard deviation 1 (mathematically, with some floating point rounding error in practice)

#### (b) K-means clustering (10 points)
Perform K-Means with K=3 using and report:
* Initialization method used and initial centroid for each cluster
* Number of iterations until convergence
* Final centroids
* Cluster variances (within-cluster sum of squares)
* Number of samples assigned to each cluster

In [ ]:
from sklearn.cluster import KMeans
from collections import Counter

cluster = KMeans(n_clusters=3, random_state=42)
cluster.fit(transformed_df[props]) # ignore Idx for now

print('Iterations before convergence: ', cluster.n_iter_)
print('Final centroids: ', cluster.cluster_centers_)

# calculate variances
transform = cluster.transform(transformed_df[props]) # gets distances to cluster


print('Samples per cluster: ', Counter(cluster.labels_))


In [ ]:
print(cluster.transform(transformed_df[props]))

#### (c) EM clustering (15 points)
Fit a Gaussian Mixture Model (GMM) with K=3 for clustering and report:
* Initialization method and initial parameters
* Convergence criterion (tolerance or max iterations)
* Covariance type (full, diagonal, spherical) with justification
* Mixture weights (𝜋_𝑘), means (𝜇_𝑘), covariances (Σ_𝑘) for each cluster
* For 3 samples, show probabilities 𝑝(𝑧 = 𝑘|𝑥)

In [21]:
from sklearn.mixture import GaussianMixture
import numpy as np

# initiate
em = GaussianMixture(
    n_components=3,
    init_params='kmeans',
    tol=1e-3,
    max_iter=100,
    covariance_type='full',
    random_state=42
)

# train
em.fit(transformed_df[props])

# print results
print("Converged:", em.converged_)
print("Iterations:", em.n_iter_)
print("Log Likelihood after Convergence:", em.lower_bound_)

print("Weights:", em.weights_)
print("Means:",em.means_)
print("Covariances:", em.covariances_)

sample_probs = em.predict_proba(transformed_df[props].iloc[:3])
print("Sample Probabilities:", np.round(sample_probs, 3))

,"n_components n_components: int, default=1The number of mixture components.",3
,"covariance_type covariance_type: {'full', 'tied', 'diag', 'spherical'}, default='full'String describing the type of covariance parameters to use.Must be one of:- 'full': each component has its own general covariance matrix.- 'tied': all components share the same general covariance matrix.- 'diag': each component has its own diagonal covariance matrix.- 'spherical': each component has its own single variance.For an example of using `covariance_type`, refer to:ref:`sphx_glr_auto_examples_mixture_plot_gmm_selection.py`.",'full'
,"tol tol: float, default=1e-3The convergence threshold. EM iterations will stop when thelower bound average gain is below this threshold.",0.001
,"reg_covar reg_covar: float, default=1e-6Non-negative regularization added to the diagonal of covariance.Allows to assure that the covariance matrices are all positive.",1e-06
,"max_iter max_iter: int, default=100The number of EM iterations to perform.",100
,"n_init n_init: int, default=1The number of initializations to perform. The best results are kept.",1
,"init_params init_params: {'kmeans', 'k-means++', 'random', 'random_from_data'}, default='kmeans'The method used to initialize the weights, the means and theprecisions.String must be one of:- 'kmeans' : responsibilities are initialized using kmeans.- 'k-means++' : use the k-means++ method to initialize.- 'random' : responsibilities are initialized randomly.- 'random_from_data' : initial means are randomly selected data points... versionchanged:: v1.1 `init_params` now accepts 'random_from_data' and 'k-means++' as initialization methods.",'kmeans'
,"weights_init weights_init: array-like of shape (n_components, ), default=NoneThe user-provided initial weights.If it is None, weights are initialized using the `init_params` method.",None
,"means_init means_init: array-like of shape (n_components, n_features), default=NoneThe user-provided initial means,If it is None, means are initialized using the `init_params` method.",None
,"precisions_init precisions_init: array-like, default=NoneThe user-provided initial precisions (inverse of the covariancematrices).If it is None, precisions are initialized using the 'init_params'method.The shape depends on 'covariance_type':: (n_components,) if 'spherical', (n_features, n_features) if 'tied', (n_components, n_features) if 'diag', (n_components, n_features, n_features) if 'full'",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to the method chosen to initialize theparameters (see `init_params`).In addition, it controls the generation of random samples from thefitted distribution (see the method `sample`).Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None


#### Report

##### Initialization Method
The Gaussian Mixture Model was initialized using the kmeans method for the initialization. The model uses 3 Gaussian components. Kmeans clsutering is first applied to the data to obtain the first parameter estimates for the misxture model. These estimates are then used as starting values for the EM algorithm. To ensure reproducibility of the cluster results random_state is set to 42.

##### Convergence Criterion
The EM algorithm was configured to run a maximum of 100 iterations. After 100 iterations the algorithm automatically stops to run to limit run time. The convergence tolerance is set to 10^-3, defining the minimum log-likelihood improvement between iterations. Once the improvement falls below this treshold the algorithm stops and the model is considered converged.The model converged after 22 iterations.

##### Covariance Type
The covariance type was set to 'full'. This allows each cluster to have its own full covariance matrix, meaning that correlations between features are considered. This allows the highest flexibility of cluster shape representation compared to diagonal or spherical covariance structures.

##### Estimated Cluster Parameters
After fitting the model, the following parameters where obtained:

Mixture Weights:
| Cluster | Weight |
|:---:|:---:|
| 1 | 0.37494468 |
| 2 | 0.38871665 |
| 3 | 0.23633867 |

The mixture weights represent the proportion of data points assigned to each cluster.

---

Cluster Means:
| Cluster | T | P | TC | SV |
|:---:|:---:|:---:|:---:|:---:|
| 1 | 1.02910681 | -0.18180898 | 0.91829939 | 1.02764859 |
| 2 | -0.61887752 | 0.74748725 | -0.27402545 | -0.80043724 |
| 3 | -0.61475394 | -0.94099044 | -1.0061545 | -0.31382122 |

The mean vectors represent the centers of the clusters in the standardized feature space. The numbers are between about -1 and +1 becuase the data was standardized using StandardScaler, which produces means at about 0 and standard deviations of about 1. Therefore, cluster centerrs are expressed in standard deviations from the global mean.

---

Covariance Matrices:

Covariance Matrix – Cluster 1

| Feature | T | P | TC | SV |
|:---:|:---:|:---:|:---:|:---:|
| T | 0.19232868 | 0.06209208 | 0.22937179 | 0.14216409 |
| P | 0.06209208 | 0.72126895 | 0.17829872 | -0.17642661 |
| TC | 0.22937179 | 0.17829872 | 0.29269885 | 0.14493227 |
| SV | 0.14216409 | -0.17642661 | 0.14493227 | 0.20417858 |

Covariance Matrix – Cluster 2

| Feature | T | P | TC | SV |
|:---:|:---:|:---:|:---:|:---:|
| T | 0.52780954 | 0.22263300 | 0.42842738 | 0.42449867 |
| P | 0.22263300 | 0.73686222 | 0.45764376 | -0.03057279 |
| TC | 0.42842738 | 0.45764376 | 0.55070818 | 0.24300361 |
| SV | 0.42449867 | -0.03057279 | 0.24300361 | 0.42664514 |

Covariance Matrix – Cluster 3

| Feature | T | P | TC | SV |
|:---:|:---:|:---:|:---:|:---:|
| T | 0.36993950 | 0.01453470 | 0.37622812 | 0.35964249 |
| P | 0.01453470 | 0.01811479 | 0.01951818 | 0.01186979 |
| TC | 0.37622812 | 0.01951818 | 0.38740695 | 0.37080814 |
| SV | 0.35964249 | 0.01186979 | 0.37080814 | 0.37789153 |

The covariance matrices describe the spread and orientation of each cluster in feature space. Larger variances indicate greater dispersion along a feature. Each Gaussian component has a 4x4 covariance matrix because the the clustering is performed in a 4 dimensional feature space.

##### Posterior Probabilities for Example Samples
To show the soft assignment properties, the posterior probablities were computed for 3 sample observations using predict_proba(). 

| Sample | Cluster 1 | Cluster 2 | Cluster 3 |
|:---:|:---:|:---:|:---:|
| 1 | 0 | 0.003 | 0.997 |
| 2 | 0 | 0.002 | 0.998 |
| 3 | 0 | 0 | 1 |

These probabilities represent the likelihood that the given observation belongs to each cluster. Unlike k-means, GMM provides soft assignments, meaning that each sample can belong to multiple clusters with different probabilities. These values indicate that the 3 samples belong strongly to the 3rd cluster.

#### (d) SOM clustering (20 points)
Train a 2D SOM and report:
* Grid size (e.g., 15x15, 20x20) with your justification
* Neighborhood function
* Learning rate schedule
* Number of iterations
* Similarity metric used

Extract clusters from SOM by clustering prototypes using K-Means: Since SOM does not directly
produce clusters, you must:
* Extract neuron weight vectors (prototypes)
* Apply K-Means with K=3 on neuron prototypes
* Assign each data point to the cluster of its BMU and report final cluster centroids and cluster sizes

In [27]:
from minisom import MiniSom
import numpy as np

# general heuristic for SOM models is that for N many samples, you want to have 5*sqrt(N) elements in the grid
# 420_000 samples -> 3240 elements ~ grid dimension 60 x 60
som = MiniSom(60, 60, 4,
              # interestingly enough the Mexican hat distribution makes some values explode to infinity
              neighborhood_function="gaussian",
              # no rhyme or reason for selection of values here, just vibes mostly lol
              learning_rate=0.75,
              decay_function="linear_decay_to_zero")

som.train(transformed_df[props].to_numpy(), num_iteration=500)

from sklearn.cluster import KMeans

somCluster = KMeans(n_clusters=3, random_state=42)
# (60 x 60 SOM grid) x (4 features) -> reshape to 3600 x 4
somCluster.fit(som.get_weights().reshape(3600, 4))
print("Final centroids: \n", somCluster.cluster_centers_)
print("Cluster sizes: \n", Counter(cluster.labels_))

Final centroids: 
 [[-0.31828362  0.10445043  0.44951897 -0.07142861]
 [-0.30629476 -0.14205045 -0.54654458 -0.12829087]
 [ 0.50542542 -0.00150732  0.0223185   0.17396663]]
Cluster sizes: 
 Counter({np.int32(0): 196274, np.int32(2): 134644, np.int32(1): 89082})


#### (e) Cluster evaluation and interpretation (20 points)

##### Silhouette score

##### Wobbe index

Wobbe indices are used to compare gases for combustion and are derived by dividing the heating value by the square root of the specific gravity. Gases with similar Wobbe indices are considered interchangeable when designing combustion systems. For the purposes of classification, we'll divide the Wobbe indices into three parts (bottom 33% = "regular", middle = "medium", top = "premium")

##### Adjusted rand index

### 2. Classification (50 points)

Split the dataset into training (70%), validation set (15%), and testing set (15%) using a fixed random seed.
All final performance metrics must be reported on the test set.

In [14]:
from sklearn.model_selection import train_test_split

# relabel Idx to Wobbe classification as mentioned above
minIdx, maxIdx = min(transformed_df['Idx']), max(transformed_df['Idx'])
threshold = (maxIdx - minIdx) / 3

def getGrade(idx):
    if idx < minIdx + threshold:
        return 1
    if idx < minIdx + 2 * threshold:
        return 2
    return 3

transformed_df['Idx'] = transformed_df['Idx'].map(getGrade)

# split data into train, val, test
training, vt = train_test_split(transformed_df, test_size=0.3, random_state=42)
validation, testing = train_test_split(vt, test_size=0.5, random_state=42)

#### (a) MLP Classifier (20 points)
Train a MLP to predict quality levels (Regular, Medium, Premium) and report:
* Topology (layers and number of neurons)
* Activation functions
* Optimizer (gradient method)
* Learning rate
* Number of epochs
* Training methods and batch size
* Regularization method if any

Evaluate on test set using confusion matrix, % accuracy, and F1-score.